In [1]:
import shutil
import pyemu
from pathlib import Path
# from pyemu.utils import VisHandler
from pyemu import vis_utils
from pyemu import os_utils
import sys
import flopy
sys.path.append('..')
from autotest.pst_from_tests import _get_port, ies_exe_path, mf6_exe_path

Executable pestpp-sen not found in /Users/brioch/Projects/dev/pyemu/examples/../bin/mac, returning None
Executable pestpp-opt not found in /Users/brioch/Projects/dev/pyemu/examples/../bin/mac, returning None
Executable pestpp-da not found in /Users/brioch/Projects/dev/pyemu/examples/../bin/mac, returning None
Executable pestpp-sqp not found in /Users/brioch/Projects/dev/pyemu/examples/../bin/mac, returning None
Executable pestpp-swp not found in /Users/brioch/Projects/dev/pyemu/examples/../bin/mac, returning None


In [2]:
m_d = Path("pst_template")
t_d = Path(".", "vis_eg", 'template')

In [3]:
if t_d.exists():
    shutil.rmtree(t_d)
shutil.copytree(m_d, t_d)

PosixPath('vis_eg/template')

In [4]:
pst = pyemu.Pst(str(t_d / "freyberg.pst"))
pst.pestpp_options['ies_num_reals'] = 10
pst.write(pst.filename, version=2)
port = _get_port()
m_d = t_d.with_name("master")
shutil.copy(shutil.which(ies_exe_path), t_d)
shutil.copy(shutil.which(mf6_exe_path), t_d)
os_utils.start_workers(t_d,"pestpp-ies","freyberg.pst",num_workers=5,
                             worker_root=t_d.parent,
                             master_dir=m_d, port=port)

noptmax:-1, npar_adj:9766, nnz_obs:53675


             pestpp-ies: a GLM iterative ensemble smoother

                   by the PEST++ development team


version: 5.2.12
binary compiled on Oct 15 2024 at 16:10:18

started at 08/19/25 15:28:17
...processing command line: ' ./pestpp-ies freyberg.pst /h :60088'
...using panther run manager in master mode using port 60088

using control file: "freyberg.pst"
in directory: "/Users/brioch/Projects/dev/pyemu/examples/vis_eg/master"
on host: "Briochs-MacBook-Pro.local"

processing control file freyberg.pst


:~-._                                                 _.-~:
: :.~^o._        ________---------________        _.o^~.:.:
 : ::.`?88booo~~~.::::::::...::::::::::::..~~oood88P'.::.:
 :  ::: `?88P .:::....         ........:::::. ?88P' :::. :
  :  :::. `? .::.            . ...........:::. P' .:::. :
   :  :::   ... ..  ...       .. .::::......::.   :::. :
   `  :' .... ..  .:::::.     . ..:::::::....:::.  `: .'
    :..    ____:::::::::.  . . ..

In [32]:
pst = pyemu.Pst(str(m_d / "freyberg.pst"))
obs = pst.observation_data
obs.loc[obs.oname=='hds', ['k', 'i', 'j']] = obs.loc[obs.oname=='hds'].obgnme.str.rsplit("_",expand=True, n=3)[[1,2,3]].values
pst.observation_data = obs
sim = flopy.mf6.MFSimulation.load(sim_ws=m_d, verbosity_level=0)
m = sim.get_model("freyberg6")
mg = m.modelgrid
mg.set_coord_info(xoff=622241.1904510253, yoff=3343617.741737109, angrot=15.0)
m.dis.xorigin = mg.xoffset
m.dis.yorigin = mg.yoffset
m.dis.angrot = mg.angrot
sim.write_simulation()

In [33]:
vh = vis_utils.VisHandler(pst, wd=m_d, crs="epsg:32614")

vminvmax:  None None
selected cellid : None
No cell selected or cellid not in map data.


In [35]:
display(vh.default_map_layout)

    'data': [{'colorscale': …

In [16]:
vh.default_unmap_layout

In [9]:
m.modelgrid

xll:622241.1904510253; yll:3343617.741737109; rotation:15.0; crs:EPSG:32614; units:meters; lenuni:2